In [0]:

import sys
from pathlib import Path

print("Python funcionando!")
print("Versão:", sys.version)
print("Diretório atual:", Path.cwd())


In [0]:

from pathlib import Path

# Caminho do documento
PDF_PATH = Path.cwd() / "docs" / "caderno_cidadania_financeira.pdf"

# Verificar se o arquivo existe
print("Caminho:", PDF_PATH)
print("Arquivo encontrado:", PDF_PATH.is_file())

if PDF_PATH.is_file():
    tamanho_mb = PDF_PATH.stat().st_size / (1024 * 1024)
    print(f"Tamanho: {tamanho_mb:.2f} MB")
else:
    print("Erro: documento não encontrado.")


In [0]:

%pip install pypdf
#A biblioteca pypdf permite ler documentos PDF e extrair o texto de suas páginas.


In [0]:

from pypdf import PdfReader

# Abrir o documento
reader = PdfReader(str(PDF_PATH))

# Armazenar texto e número de cada página
paginas = []

for numero, pagina in enumerate(reader.pages, start=1):
    texto = pagina.extract_text() or ""

    if texto.strip():
        paginas.append({
            "pagina": numero,
            "texto": texto.strip()
        })

# Exibir estatísticas
print("Total de páginas:", len(reader.pages))
print("Páginas com texto:", len(paginas))
print("Caracteres extraídos:", sum(
    len(p["texto"]) for p in paginas
))

# Visualizar um exemplo
print("\nEXEMPLO DE TEXTO EXTRAÍDO:\n")
print(paginas[0]["texto"][:1000])


## Etapa 6. Chunking, divisão do documento em trechos

In [0]:

%pip install -q langchain-text-splitters
#Pacote independente do ecossistema LangChain projetado especificamente para dividir textos extensos em partes menores e gerenciáveis (chamadas de chunks).


In [0]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuração inicial do chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    length_function=len,
    add_start_index=True
)

# Dividir o texto, preservando as páginas
chunks = []

for pagina in paginas:
    trechos = splitter.split_text(pagina["texto"])

    for indice, trecho in enumerate(trechos, start=1):
        chunks.append({
            "id": f"bcb-p{pagina['pagina']:03d}-c{indice:03d}",
            "pagina": pagina["pagina"],
            "fonte": "Caderno de Educação Financeira - BCB (2026)",
            "texto": trecho
        })

# Estatísticas
print("Total de chunks:", len(chunks))
print("Total de páginas processadas:", len(paginas))
print("Tamanho médio dos chunks:",
      round(sum(len(c["texto"]) for c in chunks) / len(chunks)))

# Exemplo
print("\nEXEMPLO DE CHUNK:\n")
print(chunks[10])


## Etapa 7. Gerar embeddings

Agora vamos transformar os textos em representações numéricas para que o RAG consiga encontrar trechos semanticamente relacionados a uma pergunta.
Suporta português, gera vetores de 384 dimensões e pode ser executado localmente, sem precisar de uma API paga.

In [0]:

%pip install -q sentence-transformers
#Transforma frases, parágrafos e documentos inteiros em representações numéricas chamadas de embeddings (vetores)

In [0]:

from sentence_transformers import SentenceTransformer

# Modelo de embeddings multilíngue
NOME_MODELO = "intfloat/multilingual-e5-small"

modelo_embeddings = SentenceTransformer(NOME_MODELO)

# Selecionar três chunks para teste
textos_teste = [
    "passage: " + chunk["texto"]
    for chunk in chunks[:3]
]

# Gerar os vetores
vetores_teste = modelo_embeddings.encode(
    textos_teste,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Modelo:", NOME_MODELO)
print("Chunks processados:", len(vetores_teste))
print("Dimensões:", vetores_teste.shape)

print("\nPrimeiros valores do vetor:")
print(vetores_teste[0][:8])


## Etapa 8. Criar o banco vetorial com ChromaDB

In [0]:

%pip install -q chromadb
# Instalação do ChromaDB
#O ChromaDB permite armazenar embeddings, documentos e metadados, além de executar consultas por similaridade.


In [0]:

import chromadb

# 1. Preparar os textos
textos = [
    "passage: " + chunk["texto"]
    for chunk in chunks
]

# 2. Gerar embeddings dos 290 chunks
vetores = modelo_embeddings.encode(
    textos,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings gerados:", vetores.shape)

# 3. Inicializar o ChromaDB em memória
cliente_chroma = chromadb.Client()

colecao = cliente_chroma.get_or_create_collection(
    name="poc_educacao_financeira",
    embedding_function=None,
    configuration={
        "hnsw": {"space": "cosine"}
    }
)

# 4. Armazenar textos, embeddings e metadados
colecao.upsert(
    ids=[c["id"] for c in chunks],
    embeddings=vetores.tolist(),
    documents=[c["texto"] for c in chunks],
    metadatas=[
        {
            "pagina": c["pagina"],
            "fonte": c["fonte"]
        }
        for c in chunks
    ]
)

# 5. Verificar o banco
print("Documentos indexados:", colecao.count())
print("Banco vetorial pronto!")


## Etapa 9. Implementar a busca semântica

In [0]:

# Pergunta para testar a recuperação
pergunta = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 1. Transformar a pergunta em embedding
vetor_pergunta = modelo_embeddings.encode(
    "query: " + pergunta,
    normalize_embeddings=True
)

# 2. Buscar os 5 chunks mais próximos
resultados = colecao.query(
    query_embeddings=[vetor_pergunta.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

# 3. Exibir os resultados
print("PERGUNTA:", pergunta)
print("\nTRECHOS RECUPERADOS:\n")

for i, (texto, metadata, distancia) in enumerate(
    zip(
        resultados["documents"][0],
        resultados["metadatas"][0],
        resultados["distances"][0]
    ),
    start=1
):
    print(f"========== RESULTADO {i} ==========")
    print(f"Página do PDF: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Fonte: {metadata['fonte']}")
    print(f"Texto:\n{texto[:600]}")
    print()


## Etapa 10. Testar uma pergunta mais específica

In [0]:

# Nova pergunta, mais específica
pergunta_teste = (
    "O que fazer diante da perda de emprego "
    "ou redução da renda familiar? "
    "Qual a importância da reserva de emergência?"
)

# Gerar embedding da pergunta
vetor_teste = modelo_embeddings.encode(
    "query: " + pergunta_teste,
    normalize_embeddings=True
)

# Recuperar os 8 resultados mais próximos
resultados_teste = colecao.query(
    query_embeddings=[vetor_teste.tolist()],
    n_results=8,
    include=["documents", "metadatas", "distances"]
)

print("PERGUNTA:", pergunta_teste)

for i, (texto, metadata, distancia) in enumerate(
    zip(
        resultados_teste["documents"][0],
        resultados_teste["metadatas"][0],
        resultados_teste["distances"][0]
    ),
    start=1
):
    print(f"\nRESULTADO {i}")
    print(f"Página: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Texto: {texto[:400]}")


## Etapa 11. Escolher o LLM para gerar as respostas

> Modelo escolhido: GPT OSS 20B

## Etapa 12. Conectar o LLM ao notebook

In [0]:

%pip install -q databricks-openai
# A parceria estratégica e a integração nativa entre a Databricks e a OpenAI (anunciada e expandida em acordos de alto impacto) levam os modelos avançados de Inteligência Artificial da OpenAI diretamente para dentro da plataforma de dados e IA da Databricks


In [0]:

from databricks_openai import DatabricksOpenAI

# Autenticação automática no Databricks
client = DatabricksOpenAI()

# Teste inicial do modelo
resposta = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "Você é um assistente de educação financeira. Responda em português do Brasil, de forma clara e objetiva."
        },
        {
            "role": "user",
            "content": "O que é uma reserva de emergência? Responda em duas frases."
        }
    ],
    max_tokens=500
)

print(resposta.choices[0].message.content)


## Etapa 13. Construir o RAG completo

![image_1794240735706.png](./image_1794240735706.png "image_1794240735706.png")

In [0]:

# ==========================================
# ETAPA 13 - RAG COMPLETO
# ==========================================

# 1. Pergunta do usuário
pergunta_rag = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 2. Gerar o embedding da pergunta
vetor_pergunta = modelo_embeddings.encode(
    "query: " + pergunta_rag,
    normalize_embeddings=True
)

# 3. Recuperar os 5 chunks mais relevantes
resultados_rag = colecao.query(
    query_embeddings=[vetor_pergunta.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

# 4. Construir o contexto com referências
trechos = []

for i, (texto, metadata) in enumerate(
    zip(
        resultados_rag["documents"][0],
        resultados_rag["metadatas"][0]
    ),
    start=1
):
    trechos.append(
        f"[TRECHO {i} | Página {metadata['pagina']} do PDF]\n"
        f"{texto}"
    )

if not trechos:
    raise ValueError("Nenhum trecho foi recuperado.")

contexto = "\n\n".join(trechos)

# 5. Instruções para o LLM
system_prompt = """
Você é um assistente de educação financeira.

Responda em português do Brasil, utilizando
exclusivamente as informações do contexto fornecido.

Regras:
1. Não invente informações.
2. Não utilize conhecimentos externos.
3. Fundamente as afirmações nos trechos recuperados.
4. Cite as páginas do PDF que sustentam a resposta.
5. Utilize referências como [Página 42 do PDF].
6. Se não houver informações suficientes,
   declare explicitamente essa limitação.
7. Não apresente recomendações como se fossem
   do Banco Central sem evidência no contexto.
"""

# 6. Enviar pergunta e contexto ao LLM
resposta_rag = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta_rag}"
            )
        }
    ],
    max_tokens=1200
)

# 7. Exibir resposta
print("PERGUNTA:")
print(pergunta_rag)

print("\nRESPOSTA DO RAG:\n")
print(resposta_rag.choices[0].message.content)

# 8. Mostrar as fontes recuperadas
print("\nPÁGINAS RECUPERADAS:")
for metadata in resultados_rag["metadatas"][0]:
    print(f"- Página {metadata['pagina']} do PDF")


## Etapa 14. Recuperação com múltiplas consultas